In [4]:
import json
from datetime import datetime

data_path = "data/sim/profiles/llm_v1/outputs/final/events_breakout.json"
data = json.load(open(data_path))

The data loaded has been "broken out". This means that certain types of events, such as value events attached to a parent event have been given their own event. If this doesn't make sense, compare to `data/sim/profiles/llm_v1/outputs/final/events_nested.json`

In [23]:
data[0:1]

[{'name': 'headache',
  'description': 'Dull pain on the left side of the head.',
  'category': 'sensation',
  'all_day': False,
  'parent_name': None,
  'dtype': 'binary',
  'source': 'journal_entry',
  'start_datetime': '2023-09-01T10:00',
  'end_datetime': '2023-09-01T11:00'}]

# Date range

In [22]:
dates = [datetime.strptime(d["start_datetime"], "%Y-%m-%dT%H:%M") for d in data]

print("First date:", min(dates).strftime("%Y-%m-%d"))
print("Last date:", max(dates).strftime("%Y-%m-%d"))

First date: 2023-09-01
Last date: 2023-12-31


# Data types

In [24]:
dtypes = set([d["dtype"] for d in data])
print("dtypes:", dtypes)

Event types: {'binary', 'numerical'}


# Events

In [34]:
print("Number of events:", len(data))

n_value_events = len([d for d in data if d.get("value")])
print("Number of value events:", n_value_events)

# number of non-value events
n_non_value_events = len(data) - n_value_events
print("Number of non-value events:", n_non_value_events)

Number of events: 6372
Number of value events: 809
Number of non-value events: 5563


## Value events

These have no category, but are distinguished by the name of the parent event.

In [37]:
# names of value events
value_event_names = set([(d["name"], d["parent_name"]) for d in data if d.get("value")])
print("Value event names:", json.dumps(list(value_event_names), indent=2))

Value event names: [
  [
    "energy",
    "daily survey"
  ],
  [
    "good",
    "sleep"
  ],
  [
    "stress",
    "daily survey"
  ],
  [
    "calories",
    "dietary intake"
  ],
  [
    "mood",
    "daily survey"
  ]
]


# Non-value Events

In [45]:
# the number of unique non-value event names
non_value_event_names = set([d["name"] for d in data if not d.get("value")])
print("Number of unique non-value event names:", len(non_value_event_names))

# list of unique non-value event names by category
non_value_event_names_by_category = {}
for d in data:
    if not d.get("value"):
        category = d["category"]
        if category not in non_value_event_names_by_category:
            non_value_event_names_by_category[category] = set()
        non_value_event_names_by_category[category].add(d["name"])
# convert sets to lists
non_value_event_names_by_category = {
    k: list(v) for k, v in non_value_event_names_by_category.items()
}
print(
    "Unique non-value event names by category:",
    json.dumps(non_value_event_names_by_category, indent=2),
)

Number of unique non-value event names: 46
Unique non-value event names by category: {
  "sensation": [
    "elbow pain",
    "shoulder pain",
    "headache",
    "stomachache"
  ],
  "food": [
    "bell peppers",
    "lemons",
    "avocado",
    "brown rice",
    "tuna",
    "ketchup",
    "salt",
    "olives",
    "basil",
    "onions",
    "garlic",
    "feta cheese",
    "cheddar cheese",
    "oregano",
    "pepper",
    "mozzarella cheese",
    "eggs",
    "tomatoes",
    "spinach",
    "lettuce",
    "olive oil",
    "beef",
    "bread"
  ],
  "dietary intake": [
    "dietary intake"
  ],
  "sleep": [
    "sleep"
  ],
  "activity": [
    "weightlifting",
    "basketball game",
    "homework due: Discrete Mathematics",
    "reading",
    "studying",
    "hiking",
    "basketball practice",
    "playing videogames",
    "party",
    "walking outside",
    "date night",
    "class: Discrete Mathematics",
    "movie night",
    "talking on the phone",
    "class: Computer Programming

# Sources and Categories

The source indicates the modality, and the category indicates a distinctive group of data that would be pulled from that modality.

In [31]:
sources = set([d["source"] for d in data])
print("sources:", sources)

categories = set([d["category"] for d in data if d["category"]])
print("categories:", categories)

# get categories for each source
categories_by_source = {}
for source in sources:
    categories_by_source[source] = list(
        set([d["category"] for d in data if d["source"] == source and d["category"]])
    )

print("\ncategories_by_source:", json.dumps(categories_by_source, indent=2))

categories: {'dietary intake', 'daily survey', 'activity', 'sleep', 'food', 'sensation'}
sources: {'survey', 'food_tracker', 'sleep_tracker', 'schedule', 'journal_entry'}

categories_by_source: {
  "survey": [
    "daily survey"
  ],
  "food_tracker": [
    "dietary intake",
    "food"
  ],
  "sleep_tracker": [
    "sleep"
  ],
  "schedule": [
    "activity"
  ],
  "journal_entry": [
    "sensation",
    "activity"
  ]
}
